
### CREATE FLAG PARAMETER

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text('incremental_flag', '0')

#It is in string type
#So we have to use '0' or '1' when we are using it 

In [0]:
incremental_flag = dbutils.widgets.get('incremental_flag')


### CREATING DIMENSION MODEL

In [0]:
%sql

SELECT * FROM parquet.`abfss://silver@adlscarproject.dfs.core.windows.net/car_sales`


##### DIM MODEL

In [0]:

df_model = spark.sql('''
                     SELECT DISTINCT(Model_ID) AS Model_ID, Model_cat FROM parquet.`abfss://silver@adlscarproject.dfs.core.windows.net/car_sales`
                     ''')
display(df_model)


In [0]:
if spark.catalog.tableExists('cars_catalog.gold.dim_model'):
    df_model_sink = spark.sql('''
    SELECT dim_model_key, MODEL_ID, Model_cat FROM cars_cata.gold.dim_model
    ''')
else:
    df_model_sink = spark.sql('''
    SELECT 1 as dim_model_key, MODEL_ID, Model_cat FROM parquet.`abfss://silver@adlscarproject.dfs.core.windows.net/car_sales` WHERE 1=0
    ''')
 

In [0]:
display(df_model_sink)

In [0]:

# Filtering new records and old records

df_filter = df_model.join(df_model_sink, df_model.Model_ID == df_model_sink.MODEL_ID, how='left').select(df_model.Model_ID, df_model.Model_cat, df_model_sink.dim_model_key)


In [0]:
display(df_filter)

In [0]:
#To see old and new data 

df_filter_old = df_filter.filter(col("dim_model_key").isNotNull())
df_filter_new = df_filter.filter(col("dim_model_key").isNull()).select("Model_ID", "Model_cat")

In [0]:
display(df_filter_new)


#### Create Surrogate Key


###### Fetch the Max Surrogate Key 

In [0]:
#Initial run - start from 1
#Incremental run - start from max(key)+1 

if incremental_flag == '0':
    max_value = 0
else:
    max_value_df = spark.sql(''' SELECT max(dim_model_key) FROM   cars_cata.gold.dim_model ''')
    max_value = max_value_df.collect()[0][0]


###### Create Surrogate Key

In [0]:
df_filter_new = df_filter_new.withColumn('dim_model_key', max_value+monotonically_increasing_id()+1)

In [0]:
display(df_filter_new)



#### Create final df

In [0]:
df_final = df_filter_new.union(df_filter_old)
display(df_final)


### SCD TYPE 1 - UPSERT

In [0]:
#if - incremental run else for initial run

if spark.catalog.tableExists('cars_cata.gold.dim_model'):
    delta_tbl = DeltaTable.forPath(spark,'abfss://gold@adlscarproject.dfs.core.windows.net/dim_model' )
    delta_tbl.alias("trg").merge(df_final.alias("src"),"trg.dim_model_key = src.dim_model_key")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
else:
    df_final.write.format("delta")\
        .mode('overwrite')\
        .option('path', 'abfss://gold@adlscarproject.dfs.core.windows.net/dim_model')\
        .saveAsTable('cars_cata.gold.dim_model')

In [0]:
%sql
SELECT * FROM cars_cata.gold.dim_model